# Mobility Insight — end-to-end EDA

This notebook tests the public APIs for Task 1 → Task 2 → Task 3 entirely in memory. 

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
root = next((path for path in (start, *start.parents) if (path / 'pyproject.toml').is_file()), None)
if root is None:
    raise FileNotFoundError('Cannot locate project root containing pyproject.toml')
for import_path in (root, root / 'src'):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
bronze = root / 'data' / 'bronze'
task_status = {}
print(f'Project root: {root}')

Project root: C:\Users\USER\Desktop\mobility-insight_task


## Task 1 — Parse and unify
Call the required `submission.parser.parse_all` API. Missing sources are passed as empty inputs and skipped according to configuration.

In [3]:
from submission.parser import parse_all

tmt_files = sorted(bronze.glob('tm*.json'))
wdmt_file = bronze / 'wdmt.json'
scgjwd_csv = bronze / 'scgjwd.csv'
scgjwd_xlsx = bronze / 'scgjwd.xlsx'
scgjwd_file = scgjwd_csv if scgjwd_csv.exists() else scgjwd_xlsx

unified_stream = parse_all(
    tmt_files=[str(path) for path in tmt_files],
    wdmt_file=str(wdmt_file) if wdmt_file.exists() else '',
    scgjwd_file=str(scgjwd_file) if scgjwd_file.exists() else '',
)
events = pd.DataFrame(unified_stream)
task_status['task1'] = f'OK — {len(events):,} events'
display(events.head())

,source,vehicle_id,recorded_at,lat,lon,speed_kmh,engine_on,odometer_km,fuel_rate_lh,engine_rpm,accel_position,brake_active
0,SCGJWD,001000000111001,2026-03-01 07:00:00+07:00,12.68,101.25,0.0,False,NaN,NaN,NaN,NaN,None
1,SCGJWD,001000000111001,2026-03-01 07:01:00+07:00,12.68,101.25,0.0,False,NaN,NaN,NaN,NaN,None
2,SCGJWD,001000000111001,2026-03-01 07:02:00+07:00,12.68,101.25,0.0,False,NaN,NaN,NaN,NaN,None
3,SCGJWD,001000000111001,2026-03-01 07:03:00+07:00,12.68,101.25,0.0,False,NaN,NaN,NaN,NaN,None
4,SCGJWD,001000000111001,2026-03-01 07:04:00+07:00,12.68,101.25,0.0,False,NaN,NaN,NaN,NaN,None


### Task 1 EDA checks

In [4]:
if events.empty:
    print('No source files were available; parse_all returned an empty list.')
else:
    summary = pd.DataFrame({
        'dtype': events.dtypes.astype(str),
        'non_null': events.notna().sum(),
        'null_pct': (events.isna().mean() * 100).round(2),
        'unique': events.nunique(dropna=True),
    })
    display(summary)
    display(events.groupby('source', observed=True).size().rename('rows').to_frame())
    assert events.recorded_at.dt.tz is not None
    assert events[['vehicle_id', 'recorded_at']].equals(
        events.sort_values(['vehicle_id', 'recorded_at'], kind='stable')
              [['vehicle_id', 'recorded_at']].reset_index(drop=True)
    )

,dtype,non_null,null_pct,unique
source,str,760,0.00,3
vehicle_id,str,760,0.00,9
recorded_at,"datetime64[us, Asia/Bangkok]",760,0.00,362
lat,float64,760,0.00,570
lon,float64,760,0.00,557
speed_kmh,float64,760,0.00,76
engine_on,object,157,79.34,2
odometer_km,float64,369,51.45,279
fuel_rate_lh,float64,7,99.08,3
engine_rpm,float64,7,99.08,6


,rows
source,
SCGJWD,150
TMT,7
WDMT,603


In [5]:
if not events.empty:
    vehicle_summary = (events.groupby(['source', 'vehicle_id'], observed=True)
        .agg(records=('recorded_at', 'size'),
             first_seen=('recorded_at', 'min'),
             last_seen=('recorded_at', 'max'),
             max_speed_kmh=('speed_kmh', 'max'))
        .reset_index())
    display(vehicle_summary)

,source,vehicle_id,records,first_seen,last_seen,max_speed_kmh
0,SCGJWD,001000000111001,30,2026-03-01 07:00:00+07:00,2026-03-01 07:29:00+07:00,0.0
1,SCGJWD,001000000111002,60,2026-03-01 07:00:00+07:00,2026-03-01 07:59:00+07:00,40.0
2,SCGJWD,001000000111003,60,2026-03-01 07:30:00+07:00,2026-03-01 08:29:00+07:00,110.0
3,TMT,tm01,2,2026-03-01 07:15:30+07:00,2026-03-01 08:15:30+07:00,0.0
4,TMT,tm02,3,2026-03-01 07:00:00+07:00,2026-03-01 09:30:00+07:00,85.0
5,TMT,tm03,2,2026-03-01 07:30:00+07:00,2026-03-01 08:30:00+07:00,95.0
6,WDMT,5กข-1122,241,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,50.0
7,WDMT,6งจ-3344,121,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,110.0
8,WDMT,7ดต-5566,241,2026-03-01 08:00:00+07:00,2026-03-01 09:00:00+07:00,90.0


## Task 2 — Build trip features
Call the required `submission.features.build_trip_features` API directly, following the same import pattern as Task 1.

In [6]:
from submission.features import build_trip_features

trip_features = build_trip_features(unified_stream)
task_status['task2'] = f'OK — {len(trip_features):,} trips'
display(trip_features.head())
display(trip_features.describe(include='all').transpose())

,trip_id,source,vehicle_id,trip_start,trip_end,record_count,segmentation_method,trip_end_reason,distance_method,trip_duration_min,...,avg_speed_kmh,max_speed_kmh,speed_variance,harsh_brake_count,harsh_accel_count,idle_ratio,time_of_day_risk,avg_fuel_rate_lh,avg_rpm,data_grade
0,34591f07433a00c5,SCGJWD,001000000111002,2026-03-01 07:00:00+07:00,2026-03-01 07:45:00+07:00,46,engine_and_gap,engine_off,haversine,45.0,...,18.282609,40.0,172.463611,<NA>,<NA>,0.244444,peak,None,None,B
1,636ebab927486fe3,SCGJWD,001000000111003,2026-03-01 07:30:00+07:00,2026-03-01 08:29:00+07:00,60,engine_and_gap,stream_end,haversine,59.0,...,94.516667,110.0,75.416389,<NA>,<NA>,0.000000,peak,None,None,B
2,1c9d8b973980ab4c,WDMT,5กข-1122,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,241,gps_gap_fallback,stream_end,odometer,120.0,...,22.373444,50.0,343.445602,<NA>,<NA>,NaN,peak,None,None,B
3,b6a43e9ff8dd7239,WDMT,6งจ-3344,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,121,gps_gap_fallback,stream_end,odometer,120.0,...,94.396694,110.0,76.520320,<NA>,<NA>,NaN,peak,None,None,B
4,9cddd819efec3559,WDMT,7ดต-5566,2026-03-01 08:00:00+07:00,2026-03-01 09:00:00+07:00,241,gps_gap_fallback,stream_end,haversine,60.0,...,62.033195,90.0,882.380641,<NA>,<NA>,NaN,peak,None,None,B


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
trip_id,5,5,34591f07433a00c5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,5,2,WDMT,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vehicle_id,5,5,001000000111002,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_start,5,NaN,NaN,NaN,2026-03-01 07:18:00+07:00,2026-03-01 07:00:00+07:00,2026-03-01 07:00:00+07:00,2026-03-01 07:00:00+07:00,2026-03-01 07:30:00+07:00,2026-03-01 08:00:00+07:00,NaN
trip_end,5,NaN,NaN,NaN,2026-03-01 08:38:48+07:00,2026-03-01 07:45:00+07:00,2026-03-01 08:29:00+07:00,2026-03-01 09:00:00+07:00,2026-03-01 09:00:00+07:00,2026-03-01 09:00:00+07:00,NaN
record_count,5.0,NaN,NaN,NaN,141.8,46.0,60.0,121.0,241.0,241.0,94.845664
segmentation_method,5,2,gps_gap_fallback,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_end_reason,5,2,stream_end,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
distance_method,5,2,haversine,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_duration_min,5.0,NaN,NaN,NaN,80.8,45.0,59.0,60.0,120.0,120.0,36.272579


### Check a specific vehicle_id

In [19]:
import pandas as pd

unified_df = pd.DataFrame(unified_stream)

# รักษา timezone และแปลงเป็น UTC+7
unified_df["recorded_at"] = (
    pd.to_datetime(unified_df["recorded_at"], utc=True)
    .dt.tz_convert("Asia/Bangkok")
)

unified_df[unified_df['vehicle_id'] == '5กข-1122']



,source,vehicle_id,recorded_at,lat,lon,speed_kmh,engine_on,odometer_km,fuel_rate_lh,engine_rpm,accel_position,brake_active
150,WDMT,5กข-1122,2026-03-01 07:00:00+07:00,13.7000,100.6000,0.0,None,12000.000,NaN,NaN,NaN,None
151,WDMT,5กข-1122,2026-03-01 07:00:30+07:00,13.7000,100.6000,0.0,None,12000.000,NaN,NaN,NaN,None
152,WDMT,5กข-1122,2026-03-01 07:01:00+07:00,13.7000,100.6000,0.0,None,12000.000,NaN,NaN,NaN,None
153,WDMT,5กข-1122,2026-03-01 07:01:30+07:00,13.7000,100.6000,0.0,None,12000.000,NaN,NaN,NaN,None
154,WDMT,5กข-1122,2026-03-01 07:02:00+07:00,13.7000,100.6000,0.0,None,12000.000,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...
386,WDMT,5กข-1122,2026-03-01 08:58:00+07:00,13.7292,100.6292,47.0,None,12000.726,NaN,NaN,NaN,None
387,WDMT,5กข-1122,2026-03-01 08:58:30+07:00,13.7294,100.6294,47.0,None,12000.733,NaN,NaN,NaN,None
388,WDMT,5กข-1122,2026-03-01 08:59:00+07:00,13.7296,100.6296,48.0,None,12000.739,NaN,NaN,NaN,None
389,WDMT,5กข-1122,2026-03-01 08:59:30+07:00,13.7298,100.6298,48.0,None,12000.746,NaN,NaN,NaN,None


## Task 3 — Driver risk scoring
Score each trip first, then aggregate grade-specific evidence per vehicle. The notebook displays the ranked feature contribution table and does not write outputs.

In [ ]:
from submission.model import DriverRiskModel

model = DriverRiskModel(method='weighted_composite')
trip_risk_scores = model.score_trips(trip_features)
scores = model.aggregate_drivers(trip_risk_scores)

robust_model = DriverRiskModel(method='robust_zscore')
robust_trip_scores = robust_model.score_trips(trip_features)
robust_scores = robust_model.aggregate_drivers(robust_trip_scores)

comparison_keys = ['source', 'vehicle_id', 'data_grade']
method_comparison = (
    scores[comparison_keys + ['final_score']].rename(columns={'final_score': 'weighted_composite'})
    .merge(
        robust_scores[comparison_keys + ['final_score']].rename(columns={'final_score': 'robust_zscore'}),
        on=comparison_keys,
        how='outer',
    )
)
method_comparison['score_delta'] = method_comparison['weighted_composite'] - method_comparison['robust_zscore']
task_status['task3'] = f'OK — 2 methods / {len(trip_risk_scores):,} trip / {len(scores):,} vehicle scores'

display(trip_risk_scores)
display(scores)
display(model.feature_contributions_)
display(robust_trip_scores)
display(method_comparison)

,trip_id,source,vehicle_id,data_grade,score_name,scoring_method,score_status,trip_start,trip_end,trip_duration_min,distance_km,trip_score,risk_points,evidence_coverage,top_risk_factor,score_version
0,34591f07433a00c5,SCGJWD,001000000111002,B,Speed Behavior Index,weighted_composite,scored,2026-03-01 07:00:00+07:00,2026-03-01 07:45:00+07:00,45.0,2.729236,97.493968,2.506032,1.0,speed_std_kmh,rule-v1-literature-informed
1,636ebab927486fe3,SCGJWD,001000000111003,B,Speed Behavior Index,weighted_composite,scored,2026-03-01 07:30:00+07:00,2026-03-01 08:29:00+07:00,59.0,9.158566,43.290000,56.710000,1.0,max_speed_kmh,rule-v1-literature-informed
2,1c9d8b973980ab4c,WDMT,5กข-1122,B,Speed Behavior Index,weighted_composite,scored,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,120.0,0.749000,93.174172,6.825828,1.0,speed_std_kmh,rule-v1-literature-informed
3,b6a43e9ff8dd7239,WDMT,6งจ-3344,B,Speed Behavior Index,weighted_composite,scored,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,120.0,3.151000,43.361983,56.638017,1.0,max_speed_kmh,rule-v1-literature-informed
4,9cddd819efec3559,WDMT,7ดต-5566,B,Speed Behavior Index,weighted_composite,scored,2026-03-01 08:00:00+07:00,2026-03-01 09:00:00+07:00,60.0,3.102738,67.016169,32.983831,1.0,speed_std_kmh,rule-v1-literature-informed


,source,vehicle_id,data_grade,score_name,scoring_method,observed_score,final_score,risk_points,trip_count,total_duration_min,total_distance_km,evidence_coverage,confidence,score_version
0,SCGJWD,001000000111002,B,Speed Behavior Index,weighted_composite,97.493968,74.582328,25.417672,1,45.0,2.729236,1.0,low,rule-v1-literature-informed
1,SCGJWD,001000000111003,B,Speed Behavior Index,weighted_composite,43.290000,65.548333,34.451667,1,59.0,9.158566,1.0,low,rule-v1-literature-informed
2,WDMT,5กข-1122,B,Speed Behavior Index,weighted_composite,93.174172,73.862362,26.137638,1,120.0,0.749000,1.0,low,rule-v1-literature-informed
3,WDMT,6งจ-3344,B,Speed Behavior Index,weighted_composite,43.361983,65.560331,34.439669,1,120.0,3.151000,1.0,low,rule-v1-literature-informed
4,WDMT,7ดต-5566,B,Speed Behavior Index,weighted_composite,67.016169,69.502695,30.497305,1,60.0,3.102738,1.0,low,rule-v1-literature-informed


,data_grade,score_name,scoring_method,rank,feature,configured_weight,mean_normalized_risk,mean_penalty_points,trips_scored,basis,rationale,references
0,B,Speed Behavior Index,weighted_composite,1,max_speed_kmh,0.5,0.280000,14.000000,5,literature_anchor_plus_design_judgment,Peak speed is the strongest observable tail-ri...,"Guillen-2024, Medarevic-2024"
1,B,Speed Behavior Index,weighted_composite,2,avg_speed_kmh,0.3,0.403786,12.113587,5,literature_anchor_plus_design_judgment,Sustained speed is observable even without eng...,Guillen-2024
2,B,Speed Behavior Index,weighted_composite,3,speed_std_kmh,0.2,0.250958,5.019155,5,smoothness_literature_plus_design_judgment,Variation is a limited GPS-only smoothness proxy.,review-paper-cite-85


,trip_id,source,vehicle_id,data_grade,score_name,scoring_method,score_status,trip_start,trip_end,trip_duration_min,distance_km,trip_score,risk_points,evidence_coverage,top_risk_factor,score_version
0,34591f07433a00c5,SCGJWD,001000000111002,B,Speed Behavior Index,robust_zscore,insufficient_cohort,2026-03-01 07:00:00+07:00,2026-03-01 07:45:00+07:00,45.0,2.729236,70.000000,30.000000,0.0,insufficient_cohort,robust-zscore-v1
1,636ebab927486fe3,SCGJWD,001000000111003,B,Speed Behavior Index,robust_zscore,insufficient_cohort,2026-03-01 07:30:00+07:00,2026-03-01 08:29:00+07:00,59.0,9.158566,70.000000,30.000000,0.0,insufficient_cohort,robust-zscore-v1
2,1c9d8b973980ab4c,WDMT,5กข-1122,B,Speed Behavior Index,robust_zscore,experimental,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,120.0,0.749000,100.000000,0.000000,1.0,none,robust-zscore-v1
3,b6a43e9ff8dd7239,WDMT,6งจ-3344,B,Speed Behavior Index,robust_zscore,experimental,2026-03-01 07:00:00+07:00,2026-03-01 09:00:00+07:00,120.0,3.151000,84.583068,15.416932,1.0,max_speed_kmh,robust-zscore-v1
4,9cddd819efec3559,WDMT,7ดต-5566,B,Speed Behavior Index,robust_zscore,experimental,2026-03-01 08:00:00+07:00,2026-03-01 09:00:00+07:00,60.0,3.102738,95.599063,4.400937,1.0,speed_std_kmh,robust-zscore-v1


,source,vehicle_id,data_grade,weighted_composite,robust_zscore,score_delta
0,SCGJWD,001000000111002,B,74.582328,70.000000,4.582328
1,SCGJWD,001000000111003,B,65.548333,70.000000,-4.451667
2,WDMT,5กข-1122,B,73.862362,75.000000,-1.137638
3,WDMT,6งจ-3344,B,65.560331,72.430511,-6.870180
4,WDMT,7ดต-5566,B,69.502695,74.266510,-4.763815


## Pipeline status

In [ ]:
display(pd.Series(task_status, name='status').rename_axis('task').to_frame())
print('EDA complete. No datasets were written by this notebook.')

,status
task,
task1,OK — 760 events
task2,OK — 5 trips
task3,OK — 2 methods / 5 trip / 5 vehicle scores


EDA complete. No datasets were written by this notebook.
